#  Resume NER Model Training

This notebook trains a **Named Entity Recognition (NER)** model using **spaCy** on the `dataset.json` resume data.

**Entities extracted:**
- `Name`
- `Skills`
- `Designation`
- `Companies worked at`
- `College Name`
- `Degree`
- `Graduation Year`
- `Location`
- `Email Address`
- `Years of Experience`


## Install Dependencies

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'spacy==3.7.4', '-q'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl'
])
print(' Dependencies installed!')

## Import Libraries

In [ ]:
import json, random, pathlib, warnings
from collections import defaultdict

import spacy
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.util import minibatch, compounding

warnings.filterwarnings('ignore')
random.seed(42)

print(f' spaCy version: {spacy.__version__}')

## Load & Parse dataset.json

In [ ]:
DATASET_PATH = pathlib.Path('dataset_augmented.json')

records = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

label_set = set(
    a['label'][0]
    for r in records
    for a in r.get('annotation', [])
    if a.get('label')
)

print(f' Loaded {len(records)} resume records')
print(f'🏷️  Entity labels ({len(label_set)}): {sorted(label_set)}')

## Convert Annotations to spaCy Training Format

In [ ]:
def build_training_data(records):
    training_data = []
    for record in records:
        text = record.get('content', '')
        if not text:
            continue
        entities = []
        for annotation in record.get('annotation', []):
            label_list = annotation.get('label', [])
            if not label_list:
                continue
            label = label_list[0]
            if not label:
                continue
            for point in annotation.get('points', []):
                start = point.get('start')
                end   = point.get('end')
                if start is None or end is None:
                    continue
                end = min(end + 1, len(text))  # inclusive → exclusive
                if start < 0 or start >= end:
                    continue
                # Trim leading/trailing whitespace from span boundaries
                while start < end and text[start].isspace():
                    start += 1
                while end > start and text[end-1].isspace():
                    end -= 1
                if start >= end:
                    continue
                entities.append((start, end, label))
        # Remove overlapping spans
        entities.sort(key=lambda x: x[0])
        clean, prev_end = [], -1
        for s, e, l in entities:
            if s >= prev_end:
                clean.append((s, e, l))
                prev_end = e
        if clean:
            training_data.append((text, {'entities': clean}))
    return training_data

training_data = build_training_data(records)
print(f' Converted {len(training_data)} records to spaCy training format')

## Train / Test Split (80 / 20)

In [ ]:
random.shuffle(training_data)
split_idx  = int(len(training_data) * 0.8)
train_data = training_data[:split_idx]
test_data  = training_data[split_idx:]
print(f' Train: {len(train_data)} | Test: {len(test_data)}')

## Build spaCy DocBin Files

In [ ]:
nlp_blank = spacy.blank('en')

def build_docbin(data, nlp):
    db, skipped = DocBin(), 0
    for text, annotations in data:
        doc  = nlp.make_doc(text)
        ents = []
        for start, end, label in annotations['entities']:
            span = doc.char_span(start, end, label=label, alignment_mode='expand')
            if span and span.text.strip():
                ents.append(span)
            else:
                skipped += 1
        doc.ents = spacy.util.filter_spans(ents)
        db.add(doc)
    return db, skipped

pathlib.Path('data').mkdir(exist_ok=True)
train_db, sk1 = build_docbin(train_data, nlp_blank)
test_db,  sk2 = build_docbin(test_data,  nlp_blank)
train_db.to_disk('data/train.spacy')
test_db.to_disk('data/dev.spacy')
print(f' DocBin saved (skipped — train: {sk1}, test: {sk2})')

## Train the NER Model

In [ ]:
all_labels = sorted(label_set)

nlp = spacy.blank('en')
ner = nlp.add_pipe('ner', last=True)
for label in all_labels:
    ner.add_label(label)

def make_examples(data, nlp):
    examples = []
    for text, annotations in data:
        doc = nlp.make_doc(text)
        ents, prev_end = [], -1
        for start, end, label in sorted(annotations['entities'], key=lambda x: x[0]):
            span = doc.char_span(start, end, label=label, alignment_mode='expand')
            if span and span.start_char >= prev_end and span.text.strip():
                ents.append((span.start_char, span.end_char, label))
                prev_end = span.end_char
        if ents:
            examples.append(Example.from_dict(doc, {'entities': ents}))
    return examples

nlp.initialize(lambda: make_examples(train_data[:10], nlp))
optimizer = nlp.get_pipe('ner').create_optimizer()

n_iter, dropout = 30, 0.3
losses_history = []

print(f' Training for {n_iter} iterations...')
print('-' * 50)
for i in range(n_iter):
    random.shuffle(train_data)
    losses = {}
    for batch in minibatch(train_data, size=compounding(4.0, 32.0, 1.001)):
        examples = make_examples(batch, nlp)
        if examples:
            nlp.update(examples, drop=dropout, losses=losses, sgd=optimizer)
    loss = round(losses.get('ner', 0), 4)
    losses_history.append(loss)
    if (i + 1) % 5 == 0 or i == 0:
        print(f'  Iteration {i+1:3d}/{n_iter}  |  NER Loss: {loss:.4f}')
print('-' * 50)
print(f' Training complete! Final loss: {losses_history[-1]:.4f}')

## Plot Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(losses_history)+1), losses_history, color='steelblue', lw=2, marker='o', ms=4)
plt.title('NER Training Loss Curve', fontsize=14, fontweight='bold')
plt.xlabel('Iteration')
plt.ylabel('NER Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print(' Loss curve saved as training_loss.png')

## Evaluate on Test Set

In [ ]:
def evaluate_ner(nlp, test_data):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for text, annotations in test_data:
        doc = nlp(text)
        pred = set((e.start_char, e.end_char, e.label_) for e in doc.ents)
        true = set()
        for start, end, label in annotations['entities']:
            span = doc.char_span(start, end, label=label, alignment_mode='expand')
            if span and span.text.strip():
                true.add((span.start_char, span.end_char, label))
        for e in pred & true:  tp[e[2]] += 1
        for e in pred - true:  fp[e[2]] += 1
        for e in true - pred:  fn[e[2]] += 1
    results = {}
    for label in sorted(set(list(tp)+list(fp)+list(fn))):
        p  = tp[label] / (tp[label]+fp[label]) if (tp[label]+fp[label]) > 0 else 0
        r  = tp[label] / (tp[label]+fn[label]) if (tp[label]+fn[label]) > 0 else 0
        f1 = 2*p*r / (p+r) if (p+r) > 0 else 0
        results[label] = {'precision': round(p,3), 'recall': round(r,3), 'f1': round(f1,3)}
    return results

eval_results = evaluate_ner(nlp, test_data)
print(f"{'Entity Label':<30} {'Precision':>10} {'Recall':>10} {'F1':>8}")
print('-' * 62)
for label, s in eval_results.items():
    print(f"{label:<30} {s['precision']:>10.3f} {s['recall']:>10.3f} {s['f1']:>8.3f}")
avg_f1 = sum(v['f1'] for v in eval_results.values()) / len(eval_results)
print('-' * 62)
print(f"{'AVERAGE F1':<30} {avg_f1:>28.3f}")

##  Test Prediction on a Sample Resume

In [ ]:
sample_text = test_data[0][0] if test_data else records[0]['content']
doc = nlp(sample_text[:600])

grouped = defaultdict(list)
for ent in doc.ents:
    grouped[ent.label_].append(ent.text.strip())

print('🏷️  Extracted Entities:')
print('-' * 50)
for label, values in sorted(grouped.items()):
    print(f'  {label:<25}: {values[:3]}')

##  Save the Trained Model

In [ ]:
MODEL_PATH = pathlib.Path('ner_model')
nlp.to_disk(MODEL_PATH)
print(f' Model saved to: {MODEL_PATH.resolve()}')

# Verify by loading back
loaded = spacy.load(MODEL_PATH)
test_doc = loaded('Abhishek Jha Application Development Associate Accenture Bengaluru')
print(f' Verification: {[(e.text, e.label_) for e in test_doc.ents]}')

## Use the Model (Helper Function)

In [ ]:
def parse_resume_ner(resume_text: str, model_path: str = 'ner_model') -> dict:
    """Parse raw resume text and return structured entities."""
    nlp = spacy.load(model_path)
    doc = nlp(resume_text)
    label_map = {
        'Name': 'name', 'Skills': 'skills', 'Designation': 'designations',
        'Companies worked at': 'companies', 'College Name': 'colleges',
        'Degree': 'degrees', 'Graduation Year': 'graduation_years',
        'Location': 'locations', 'Email Address': 'emails',
        'Years of Experience': 'experience'
    }
    result = defaultdict(list)
    for ent in doc.ents:
        key = label_map.get(ent.label_, ent.label_.lower().replace(' ', '_'))
        text = ent.text.strip()
        if text and text not in result[key]:
            result[key].append(text)
    return dict(result)

# Quick demo
output = parse_resume_ner(records[0]['content'][:800])
print(' Structured Resume Output:')
for key, values in output.items():
    print(f'  {key:<20}: {values[:2]}')
print('\n Model is ready! Load with: nlp = spacy.load("ner_model")')